In [1]:
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics.pairwise import cosine_similarity
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

In [2]:
data = pd.read_csv('/kaggle/input/psychiatric-drug/psychiatric_drug.csv')
data.sample(5) 

,Unnamed: 0,drug_name,date,age,gender,time_on_drug,reviewer_type,condition,rating_overall,rating_effectiveness,rating_ease_of_use,rating_satisfaction,text
2964,2964,Bupropion HCl Oral,7/3/2012,19-24,Female,less than 1 month,Patient,Other,2.7,3,3,2,I just started taking the medication for my zo...
44302,44302,Alprazolam Oral,3/14/2008,25-34,Female,1 to less than 2 years,Patient,Panic Disorder,5.0,5,5,5,having anxiety&panic disorder on a daily basis...
27076,27076,Lexapro Oral,4/25/2012,65-74,Female,2 to less than 5 years,Caregiver,Depression,4.0,4,4,4,The use of a daily 10mg dose of Lexapro has be...
34909,34909,Paxil Oral,8/28/2010,25-34,Female,1 to 6 months,Patient,Panic Disorder,4.0,4,4,4,I took this drug when I was younger for depres...
35233,35233,Paxil Oral,6/21/2008,55-64,Female,5 to less than 10 years,Patient,Repeated Episodes of Anxiety,5.0,5,5,5,"I have been on Paxil for 8 years, and recently..."


In [3]:
profile = ProfileReport(data, title="Pandas Profiling Report")

In [4]:
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [5]:
data = data.dropna()

In [6]:
data.drop(columns=['Unnamed: 0', 'date','time_on_drug','reviewer_type','rating_effectiveness','rating_ease_of_use','rating_satisfaction'], inplace=True)

<ipython-input-6-5a434133cc66>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.drop(columns=['Unnamed: 0', 'date','time_on_drug','reviewer_type','rating_effectiveness','rating_ease_of_use','rating_satisfaction'], inplace=True)


In [7]:
data.sample(2)

,drug_name,age,gender,condition,rating_overall,text
7931,Citalopram Oral,25-34,Male,Panic Disorder,1.0,high blood pressure as the side effect
4196,Wellbutrin SR Oral,45-54,Female,Stop Smoking,4.7,used for 2 months and quit smoking the day I b...


In [8]:
# Function to get sentiment
def get_sentiment(review):
    analysis = TextBlob(review)
    if analysis.sentiment.polarity > 0:
        return 'positive'
    elif analysis.sentiment.polarity == 0:
        return 'neutral'
    else:
        return 'negative'

# Create a new sentiment column
data['sentiment'] = data['text'].apply(get_sentiment)

<ipython-input-8-2f131cb1f6bf>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['sentiment'] = data['text'].apply(get_sentiment)


In [9]:
data.sample(5)

,drug_name,age,gender,condition,rating_overall,text,sentiment
30333,Lexapro Oral,45-54,Female,Repeated Episodes of Anxiety,5.0,i have stoped taking the pill but i know i nee...,negative
51351,Lorazepam Oral,55-64,Male,Anxious,5.0,good,positive
56038,Cymbalta Oral,25-34,Male,Repeated Episodes of Anxiety,2.0,I was prescribed this drug after multiple conc...,negative
36932,Fluoxetine,35-44,Female,Depression,5.0,Prozac is a miracle drug for me. It keeps me ...,positive
15012,Zoloft Oral,45-54,Female,Depression,5.0,Zoloft has worked wonders for me. I had stoppe...,positive


In [10]:
reviews = data['text']
labels = data['sentiment']
ages = data['age']
sexes = data['gender']
conditions = data['condition']
drugs = data['drug_name']

In [11]:
# TF-IDF Vectorization for reviews
vectorizer = TfidfVectorizer(stop_words='english')
X_reviews = vectorizer.fit_transform(reviews)
y_reviews = labels

In [12]:
X_train_reviews, X_test_reviews, y_train_reviews, y_test_reviews = train_test_split(X_reviews, y_reviews, test_size=0.2, random_state=42)

In [13]:
smote = SMOTE(random_state=42)

In [14]:
X_train_smote_reviews, y_train_smote_reviews = smote.fit_resample(X_train_reviews, y_train_reviews)

In [15]:
svm_model = SVC(kernel='linear', class_weight='balanced')
svm_model.fit(X_train_smote_reviews, y_train_smote_reviews)

SVC(class_weight='balanced', kernel='linear')

In [16]:
sentiment_predictions = svm_model.predict(X_test_reviews)

In [17]:
accuracy = accuracy_score(y_test_reviews, sentiment_predictions)
print(accuracy)

0.8323542210617929


In [18]:
sex_encoded = pd.get_dummies(sexes, prefix='sex')
condition_encoded = pd.get_dummies(conditions, prefix='condition')
age_encoded = pd.get_dummies(ages, prefix='age')

In [19]:
user_profiles = pd.concat([age_encoded, sex_encoded, condition_encoded], axis=1)

In [20]:
def cosine_similarity_batch(user_profiles, batch_size=500):
    num_users = user_profiles.shape[0]
    user_similarity = np.zeros((num_users, num_users))

    for i in range(0, num_users, batch_size):
        batch_end = min(i + batch_size, num_users)
        batch_profiles = user_profiles[i:batch_end]

        # Calculate similarity for the current batch against all profiles
        batch_similarity = cosine_similarity(batch_profiles, user_profiles)

        # Store the similarity results in the main similarity matrix
        user_similarity[i:batch_end] = batch_similarity

    return user_similarity

# Usage:
user_similarity = cosine_similarity_batch(user_profiles)

In [21]:
def recommend_drugs_for_user(age, sex, condition, user_profiles, sentiment_predictions, drugs):
    import pandas as pd
    import numpy as np
    from sklearn.metrics.pairwise import cosine_similarity

    # Create a profile for the input user
    sex_encoded = pd.get_dummies([sex], prefix='sex')
    condition_encoded = pd.get_dummies([condition], prefix='condition')
    age_encoded = pd.get_dummies([age], prefix='age')
    user_profile = pd.concat([age_encoded, sex_encoded, condition_encoded], axis=1)

    # Ensure the new user profile has the same columns as the existing user profiles
    user_profile = user_profile.reindex(columns=user_profiles.columns, fill_value=0)

    # Calculate similarity between the input user and existing users
    similarity = cosine_similarity(user_profile, user_profiles)[0]

    # Get the indices of the most similar users
    similar_users = np.argsort(similarity)[::-1]

    # Generate drug recommendations based on similar users and sentiment predictions
    recommended_drugs = []
    unique_drugs = set()  # To track unique drugs

    for similar_user in similar_users:
        # Check if the index is within bounds
        if similar_user < len(sentiment_predictions) and similar_user < len(drugs):
            # Access `drugs` by position (iloc) instead of index
            if sentiment_predictions[similar_user] == 'positive':
                drug = drugs.iloc[similar_user]
                if drug not in unique_drugs:  # Add only if unique
                    recommended_drugs.append(drug)
                    unique_drugs.add(drug)

                # Stop if we already have 5 unique drugs
                if len(recommended_drugs) == 5:
                    break

    return recommended_drugs


In [22]:
age = '45-54'
sex = 'male'
condition = 'depression'

recommendations = recommend_drugs_for_user(age, sex, condition, user_profiles,sentiment_predictions, drugs)
print("Recommended Drugs:", recommendations)

Recommended Drugs: ['Paxil CR Oral', 'Geodon Oral', 'Perphenazine-amitriptyline Oral', 'Chlordiazepoxide Oral', 'Citalopram Oral']


In [23]:
import joblib
joblib.dump(svm_model, 'svm_model.pkl')

['svm_model.pkl']